<a href="https://colab.research.google.com/github/Vdivyajaswanthi123/ai-mentor-portfolio/blob/main/Day9_Sprint_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import os

# Remove old key
del os.environ['GEMINI_API_KEY']

# Set new key securely
import getpass
os.environ['GEMINI_API_KEY'] = getpass.getpass('Enter new Gemini API key: ')

# Confirm it's updated
print("Key set:", os.environ['GEMINI_API_KEY'][:10], "...")

Enter new Gemini API key: ··········
Key set: AQ.Ab8RN6J ...


In [ ]:
!pip install -U ddgs

In [ ]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

In [42]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_883/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [43]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '545667a2-8c7e-4f25-beb8-9cf5482274d0', 'type': 'tool_call'}]

[2] ToolMessage
    Content: India’s largest IT services firm, Tata Consultancy Services (TCS), has made 25,000 job offers to fresh graduates for FY2026-27, signaling a cautious yet continued commitment to campus hiring ... Tata Consultancy Services reported its consolidated financial results according to Ind AS and IFRS, for t

[3] AIMessage
    Content: [{'type': 'text', 'text': 'TCS has made 25,000 job offers to fresh graduates for FY2026-27. The company plans to onboard around 40,000 freshers annually.', 'extras': {'signature': 'CtAJAQw51sfvMaoj9Gl9T2ehy6tN16fNks7YjOMM1xdqw1KZvOouLbTC5pLZ5ZN5y8H72fBNd7pDdNwKulBdqtXRKNc8UJxRRSeWudr5DvyY9LcZ9KIZFIl


In [ ]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')

In [21]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [22]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))
# Expected: 'aws, spring boot'

aws, spring boot


In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))
# Expected: low score (~3-4/10), rationale about lack of specificity / cultural fit

**Score: 2/10**

**Rationale:** Entirely self-serving and generic, demonstrating no specific interest in TCS Digital's work, mission, or how the candidate aligns with the role beyond personal gain.


In [24]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_883/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [44]:
import pathlib
lines = pathlib.Path('./sample_data/sample_resumes.txt').read_text().strip().split('\n')

for i, line in enumerate(lines[:2]):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {line}')
    print(f'{"="*70}')

    msg = (f"Here is a student profile: {line}. "
           f"Plan 3 mock interview questions, score one sample answer, "
           f"and tell what skills they need to be a strong fit.")

    result = agent.invoke({'messages': [('user', msg)]}, config={'recursion_limit': 10})

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')


Student 1: Ravi Kumar

  [0] HumanMessage
      Here is a student profile: Ravi Kumar. Plan 3 mock interview questions, score one sample answer, and tell what skills they need to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': "Please tell me what role Ravi Kumar is interviewing for. Once I have that information, I can create relevant mock interview questions, tell you what skills are needed for the role, and score a sample answer (which you'll need to provide).", 'extras': {'signature': 'CrIFAQw

Student 2: ravi.kumar@example.com | +91-9876543210 | Hyderabad

  [0] HumanMessage
      Here is a student profile: ravi.kumar@example.com | +91-9876543210 | Hyderabad. Plan 3 mock interview questions, score one sample answer, and tell what skills they need to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Here are 3 mock interview questions:\n\n1.  "Tell me about a time you faced a significant challenge in a project or task. How did you approach it, and

In [45]:
# Pass a bad URL — see how agent recovers
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I am sorry, I cannot fetch the JD from the provided URL. I can only access information available via web search. As this URL does not exist, I am unable to find the job description or extract the must-have skills.', 'extras': {'signature': 'CqgHAQw51scvMQmywQoIdwFYSe3jumie
